In [9]:
import pandas as pd
import numpy as np
from numpy import *
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score, confusion_matrix, classification_report
from itertools import product
import matplotlib.pyplot as plt
import seaborn as sns
import sys

df = pd.read_csv("data/forestfires.csv")
y_label = 'area_bin'
df

,X,Y,month,day,FFMC,DMC,DC,ISI,temp,RH,wind,rain,area
0,7,5,mar,fri,86.2,26.2,94.3,5.1,8.2,51,6.7,0.0,0.00
1,7,4,oct,tue,90.6,35.4,669.1,6.7,18.0,33,0.9,0.0,0.00
2,7,4,oct,sat,90.6,43.7,686.9,6.7,14.6,33,1.3,0.0,0.00
3,8,6,mar,fri,91.7,33.3,77.5,9.0,8.3,97,4.0,0.2,0.00
4,8,6,mar,sun,89.3,51.3,102.2,9.6,11.4,99,1.8,0.0,0.00
...,...,...,...,...,...,...,...,...,...,...,...,...,...
512,4,3,aug,sun,81.6,56.7,665.6,1.9,27.8,32,2.7,0.0,6.44
513,2,4,aug,sun,81.6,56.7,665.6,1.9,21.9,71,5.8,0.0,54.29
514,7,4,aug,sun,81.6,56.7,665.6,1.9,21.2,70,6.7,0.0,11.16
515,1,4,aug,sat,94.4,146.0,614.7,11.3,25.6,42,4.0,0.0,0.00


In [5]:
df_pre = df.copy()
df_pre['month'] = df_pre['month'].astype('category').cat.codes
df_pre['day'] = df_pre['day'].astype('category').cat.codes
df_pre['area'] = ma.log(df_pre['area'].values).filled(0)
area_labels = [1, 2, 3, 4, 5, 6]
df_pre['area_bin'] = pd.cut(df_pre['area'], include_lowest=True, bins=6, labels=area_labels)
df_pre.drop(columns=['area'], inplace=True)
df_pre

,X,Y,month,day,FFMC,DMC,DC,ISI,temp,RH,wind,rain,area_bin
0,7,5,7,0,86.2,26.2,94.3,5.1,8.2,51,6.7,0.0,2
1,7,4,10,5,90.6,35.4,669.1,6.7,18.0,33,0.9,0.0,2
2,7,4,10,2,90.6,43.7,686.9,6.7,14.6,33,1.3,0.0,2
3,8,6,7,0,91.7,33.3,77.5,9.0,8.3,97,4.0,0.2,2
4,8,6,7,3,89.3,51.3,102.2,9.6,11.4,99,1.8,0.0,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...
512,4,3,1,3,81.6,56.7,665.6,1.9,27.8,32,2.7,0.0,3
513,2,4,1,3,81.6,56.7,665.6,1.9,21.9,71,5.8,0.0,5
514,7,4,1,3,81.6,56.7,665.6,1.9,21.2,70,6.7,0.0,4
515,1,4,1,2,94.4,146.0,614.7,11.3,25.6,42,4.0,0.0,2


In [6]:
from imblearn.over_sampling import SMOTE, ADASYN
from tensorflow.keras import backend as k
df_x = df_pre.drop(columns=[y_label])
df_y = df_pre[y_label]
oversample = ADASYN(sampling_strategy='all', n_neighbors=2)
df_x_smote, df_y_smote = oversample.fit_resample(df_x, df_y)
df_y_smote

0       2
1       2
2       2
3       2
4       2
       ..
1805    6
1806    6
1807    6
1808    6
1809    6
Name: area_bin, Length: 1810, dtype: category
Categories (6, int64): [1 < 2 < 3 < 4 < 5 < 6]

In [7]:
x_train, x_test, y_train, y_test = train_test_split(df_x, df_y, test_size=0.3, random_state=1)
x_train_smote, x_test_smote, y_train_smote, y_test_smote = train_test_split(df_x_smote, df_y_smote, test_size=0.3, random_state=1)
y_test.value_counts()

2    95
3    31
4    19
5     7
1     3
6     1
Name: area_bin, dtype: int64

In [8]:
def get_sensitivity(cm, label):
    labels = [1,2,3,4,5,6]
    #cm = confusion_matrix(y_test, y_pred, labels=labels)
    #print("{}/{}".format(cm[label - 1, label - 1], sum(cm[label - 1])))
    return cm[label - 1, label - 1] / sum(cm[label - 1])

def get_specificity(cm, label):
    indexes = [0,1,2,3,4,5]
    #cm = confusion_matrix(y_test, y_pred, labels=[1,2,3,4,5,6])
    tn = sum(np.diag(cm)) - cm[label - 1, label - 1]
    fp = sum([cm[i][label - 1] for i in indexes]) - cm[label - 1, label - 1]
    #print("tn:{}, fp:{}".format(tn, fp))
    return tn / (tn + fp)

def display_sens_spec(y_pred, y_test):
    labels = [1,2,3,4,5,6]
    cm = confusion_matrix(y_test, y_pred, labels=labels)
    result = pd.DataFrame(columns=['label', 'sensitivity', 'specificity'])
    for idx, label in enumerate(labels):
        result.loc[idx] = [label, get_sensitivity(cm, label), get_specificity(cm, label)]
    result.set_index('label', inplace=True)
    return result    

In [11]:
knn_clf = KNeighborsClassifier(n_neighbors=5)
knn_clf.fit(x_train, y_train)
y_pred = knn_clf.predict(x_test)
print("Accuracy:", accuracy_score(y_test, y_pred))
print(confusion_matrix(y_test, y_pred, labels=[1,2,3,4,5,6]))
print(display_sens_spec(y_pred, y_test))

Accuracy: 0.5
[[ 0  2  1  0  0  0]
 [ 3 73 18  1  0  0]
 [ 1 25  5  0  0  0]
 [ 0 17  2  0  0  0]
 [ 0  5  2  0  0  0]
 [ 0  1  0  0  0  0]]
       sensitivity  specificity
label                          
1.0       0.000000     0.951220
2.0       0.768421     0.090909
3.0       0.161290     0.760417
4.0       0.000000     0.987342
5.0       0.000000     1.000000
6.0       0.000000     1.000000


In [12]:
knn_clf = KNeighborsClassifier(n_neighbors=5)
knn_clf.fit(x_train_smote, y_train_smote)
y_pred = knn_clf.predict(x_test_smote)
print("Accuracy:", accuracy_score(y_test_smote, y_pred))
print(confusion_matrix(y_test_smote, y_pred, labels=[1,2,3,4,5,6]))
print(display_sens_spec(y_pred, y_test_smote))

Accuracy: 0.7403314917127072
[[92  0  2  0  0  3]
 [ 7 28 24 19 16  3]
 [ 4 16 42 10  0  6]
 [ 1  3  7 74  1  0]
 [ 1  4  4  5 92  0]
 [ 0  3  1  1  0 74]]
       sensitivity  specificity
label                          
1.0       0.948454     0.959752
2.0       0.288660     0.935000
3.0       0.538462     0.904523
4.0       0.860465     0.903581
5.0       0.867925     0.948012
6.0       0.936709     0.964706


In [19]:
for n in range(1, 50):
    knn_clf = KNeighborsClassifier(n_neighbors=n)
    knn_clf.fit(x_train, y_train)
    y_pred = knn_clf.predict(x_test)
    print("Accuracy with n_neighbors {}: {}".format(n, accuracy_score(y_test, y_pred)))

Accuracy with n_neighbors 1: 0.391025641025641
Accuracy with n_neighbors 2: 0.47435897435897434
Accuracy with n_neighbors 3: 0.4423076923076923
Accuracy with n_neighbors 4: 0.5
Accuracy with n_neighbors 5: 0.5
Accuracy with n_neighbors 6: 0.5384615384615384
Accuracy with n_neighbors 7: 0.5256410256410257
Accuracy with n_neighbors 8: 0.5128205128205128
Accuracy with n_neighbors 9: 0.532051282051282
Accuracy with n_neighbors 10: 0.5384615384615384
Accuracy with n_neighbors 11: 0.5641025641025641
Accuracy with n_neighbors 12: 0.5512820512820513
Accuracy with n_neighbors 13: 0.5512820512820513
Accuracy with n_neighbors 14: 0.5769230769230769
Accuracy with n_neighbors 15: 0.5384615384615384
Accuracy with n_neighbors 16: 0.5641025641025641
Accuracy with n_neighbors 17: 0.5641025641025641
Accuracy with n_neighbors 18: 0.5769230769230769
Accuracy with n_neighbors 19: 0.5769230769230769
Accuracy with n_neighbors 20: 0.5897435897435898
Accuracy with n_neighbors 21: 0.5961538461538461
Accuracy wi

In [21]:
for n in range(1, 50):
    knn_clf = KNeighborsClassifier(n_neighbors=n)
    knn_clf.fit(x_train_smote, y_train_smote)
    y_pred = knn_clf.predict(x_test)
    print("Accuracy with n_neighbors {}: {}".format(n, accuracy_score(y_test, y_pred)))

Accuracy with n_neighbors 1: 0.8461538461538461
Accuracy with n_neighbors 2: 0.8076923076923077
Accuracy with n_neighbors 3: 0.6346153846153846
Accuracy with n_neighbors 4: 0.6217948717948718
Accuracy with n_neighbors 5: 0.5512820512820513
Accuracy with n_neighbors 6: 0.5
Accuracy with n_neighbors 7: 0.4551282051282051
Accuracy with n_neighbors 8: 0.4230769230769231
Accuracy with n_neighbors 9: 0.3782051282051282
Accuracy with n_neighbors 10: 0.36538461538461536
Accuracy with n_neighbors 11: 0.34615384615384615
Accuracy with n_neighbors 12: 0.3333333333333333
Accuracy with n_neighbors 13: 0.3333333333333333
Accuracy with n_neighbors 14: 0.32051282051282054
Accuracy with n_neighbors 15: 0.2692307692307692
Accuracy with n_neighbors 16: 0.27564102564102566
Accuracy with n_neighbors 17: 0.2948717948717949
Accuracy with n_neighbors 18: 0.28846153846153844
Accuracy with n_neighbors 19: 0.26282051282051283
Accuracy with n_neighbors 20: 0.2692307692307692
Accuracy with n_neighbors 21: 0.262820